# Fine-tuning DiffWater under two noise schedules: Run 0, Run A, Run B
Group 25 · Foundation Models and Generative AI · Noise Schedule Ablation

One run per Colab session (T4 GPU). In the launch cell set `RUN` to `"A"` (original schedule), `"B"` (schedule raised to the power X = 2.00, set at the top of `finetune.py`) or `"0"` (pretrained model, no fine-tuning). Everything else is identical for the three runs. Run the cells in order; do not use "Run all".

Needs: `finetune.py` in `/content` (or in `My Drive/FoundationModel-Assignment/`), and the pretrained checkpoint `I390000_E78_gen.pth` in `My Drive/DW/LSUI/`.

This is a cleaned copy of the notebook we ran: the code of the setup, data, check and launch cells is unchanged; the tutorial text, quizzes and the optional benchmark and smoke-test cells were removed.

## 1. Setup (Drive, DiffWater code at the pinned commit, libraries)

In [ ]:
import os, sys, subprocess, glob, json, random, hashlib, shutil, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run this cell again."

from google.colab import drive
drive.mount("/content/drive")

# ---- paths. Edit only if your Drive layout differs ----
A_DIR = "/content/drive/MyDrive/FoundationModel-Assignment"      # outputs and shared files live here
CKPT_GEN = "/content/drive/MyDrive/DW/LSUI/I390000_E78_gen.pth"  # the pretrained checkpoint (same as Run 0)
SCRIPT = "/content/finetune.py"                                  # uploaded to Colab's local disk
# ------------------------------------------------------

# create the folders this account may not have yet (a fresh Google account has none of them)
os.makedirs(A_DIR, exist_ok=True)
os.makedirs(f"{A_DIR}/runs", exist_ok=True)
os.makedirs(os.path.dirname(CKPT_GEN), exist_ok=True)

if not os.path.exists(SCRIPT) and os.path.exists(f"{A_DIR}/finetune.py"):
    shutil.copy(f"{A_DIR}/finetune.py", SCRIPT)                  # fallback: a copy kept in Drive
assert os.path.exists(SCRIPT), "Upload finetune.py to /content (left sidebar > Files > Upload)"

assert os.path.exists(CKPT_GEN) and os.path.getsize(CKPT_GEN) > 5e8, (
    f"checkpoint missing or too small: {CKPT_GEN}\n"
    "This Google account needs its own copy: open the authors' folder (link in the notes above Step 0), "
    "right-click LSUI/I390000_E78_gen.pth -> Make a copy, and move it to My Drive/DW/LSUI/ in THIS account.")
CKPT_PREFIX = CKPT_GEN[:-len("_gen.pth")]       # DiffWater's loader wants the path WITHOUT "_gen.pth"

REPO_ROOT = "/content/DiffWater"; REPO_DIR = f"{REPO_ROOT}/DiffWater"      # the real code is in the nested folder
if not os.path.isdir(REPO_ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/Guan-MS/DiffWater.git", REPO_ROOT], check=True)
print("DiffWater commit:", subprocess.run(["git", "-C", REPO_ROOT, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
!pip -q install tensorboardX scikit-image lmdb pyyaml kagglehub pandas
print("checkpoint:", CKPT_PREFIX, "| script:", SCRIPT)

## 2. Data: held-out test pairs, training pairs, duplicate removal, fingerprints

In [ ]:
from PIL import Image
import re, kagglehub

KAGGLE_ROOT = kagglehub.dataset_download("pamuduranasinghe/euvp-dataset")
EUVP = f"{KAGGLE_ROOT}/EUVP"
SEED, N_TEST, N_TRAIN = 1234, 30, 800      # N_TRAIN: size of the fine-tuning subset (identical for A and B)
NEAR_DUP_HAMMING = 6                       # dHash bits (of 64) at or below which two images count as near-duplicates
EXT = (".jpg", ".jpeg", ".png", ".bmp")

def index(d):
    return {os.path.splitext(f)[0]: os.path.join(d, f) for f in sorted(os.listdir(d)) if f.lower().endswith(EXT)}
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
save256 = lambda s, d: Image.open(s).convert("RGB").resize((256, 256), Image.Resampling.LANCZOS).save(d)
def dhash(path):                           # 64-bit difference hash
    a = np.asarray(Image.open(path).convert("L").resize((9, 8), Image.Resampling.LANCZOS), dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).ravel()

# ---- 1. every paired image of the declared subsets ----
pairs = []
for sub in ("underwater_dark", "underwater_scenes"):
    a, b = index(f"{EUVP}/Paired/{sub}/trainA"), index(f"{EUVP}/Paired/{sub}/trainB")
    stems = sorted(set(a) & set(b))
    print(f"{sub}: trainA {len(a)}  trainB {len(b)}  matched by name {len(stems)}")
    assert len(stems) > 0.9 * min(len(a), len(b)), "pairing by filename failed - send me the folder listing"
    pairs += [(sub, s, a[s], b[s]) for s in stems]

# ---- 2. fingerprints: exact (MD5) and perceptual (dHash) ----
t0 = time.time()
fp = {p[2]: (md5(p[2]), dhash(p[2])) for p in pairs}
print(f"fingerprinted {len(fp)} images in {time.time() - t0:.0f}s")

# ---- 3. test pairs: Dark only, seeded, no exact duplicates among themselves ----
dark = [p for p in pairs if p[0] == "underwater_dark"]
random.Random(SEED).shuffle(dark)
test, seen = [], set()
for p in dark:
    if fp[p[2]][0] in seen: continue
    seen.add(fp[p[2]][0]); test.append(p)
    if len(test) == N_TEST: break
test = sorted(test, key=lambda p: (p[0], p[1]))

# ---- 4. training pool: everything else minus exact and near duplicates of any test image ----
test_md5 = {fp[p[2]][0] for p in test}; test_keys = {p[2] for p in test}
test_h = np.stack([fp[p[2]][1] for p in test])                  # (N_TEST, 64)
pool, n_exact, n_near = [], 0, 0
for p in pairs:
    if p[2] in test_keys: continue
    m, h = fp[p[2]]
    if m in test_md5: n_exact += 1; continue
    if (test_h != h).sum(axis=1).min() <= NEAR_DUP_HAMMING: n_near += 1; continue
    pool.append(p)
print(f"training pool: {len(pool)} pairs | removed: {n_exact} exact duplicates, {n_near} near-duplicates of a test image")
imnet = lambda s: re.search(r"_n\d{8}_", s) is not None
print(f"names that look ImageNet-derived: test {sum(imnet(p[1]) for p in test)}/{len(test)}, pool {sum(imnet(p[1]) for p in pool)}/{len(pool)}")

# ---- 5. identical split in every session ----
split = [[p[0], p[1]] for p in test]
os.makedirs(A_DIR, exist_ok=True)
sf = f"{A_DIR}/test_split_dark30.json"
if os.path.exists(sf):
    assert json.load(open(sf)) == split, "this session built a DIFFERENT test split than the one saved on Drive!"
    print("test split identical to the one saved on Drive")
else:
    json.dump(split, open(sf, "w")); print("saved test split to", sf)
print("split fingerprint:", hashlib.md5(json.dumps(split).encode()).hexdigest()[:12], " <- must be identical in every account/session")

# ---- 6. write the 256x256 folders ----
OUT_TEST = "/content/euvp_test"; OUT_TRAIN = "/content/euvp_train"
for root in (OUT_TEST, OUT_TRAIN):
    shutil.rmtree(root, ignore_errors=True)
    for sub in ("input_256", "target_256"): os.makedirs(f"{root}/{sub}")
for i, (sub, s, pa, pb) in enumerate(test):
    save256(pa, f"{OUT_TEST}/input_256/{i:04d}_{sub}_{s}.png"); save256(pb, f"{OUT_TEST}/target_256/{i:04d}_{sub}_{s}.png")
random.Random(SEED).shuffle(pool)
for i, (sub, s, pa, pb) in enumerate(pool[:N_TRAIN]):
    save256(pa, f"{OUT_TRAIN}/input_256/{i:05d}_{sub}_{s}.png"); save256(pb, f"{OUT_TRAIN}/target_256/{i:05d}_{sub}_{s}.png")
print("wrote", len(os.listdir(f"{OUT_TRAIN}/input_256")), "train pairs and", len(os.listdir(f"{OUT_TEST}/input_256")), "test pairs")

fig, ax = plt.subplots(4, 4, figsize=(12, 12))
for blk, (root, title) in enumerate(((OUT_TRAIN, "TRAIN"), (OUT_TEST, "TEST"))):
    for j, fn in enumerate(sorted(os.listdir(f"{root}/input_256"))[:4]):
        ax[2*blk][j].imshow(Image.open(f"{root}/input_256/{fn}")); ax[2*blk+1][j].imshow(Image.open(f"{root}/target_256/{fn}"))
        ax[2*blk][j].axis("off"); ax[2*blk+1][j].axis("off")
    ax[2*blk][0].set_title(f"{title}: top row degraded INPUT | bottom row REFERENCE", loc="left")
plt.tight_layout(); plt.show()

In [ ]:
tr = sorted(os.listdir("/content/euvp_train/input_256"))
print("train pairs:", len(tr), "| dark:", sum("_underwater_dark_" in f for f in tr), "| scenes:", sum("_underwater_scenes_" in f for f in tr))
print("train fingerprint:", hashlib.md5("".join(tr).encode()).hexdigest()[:12], " <- must be identical in the A and B accounts")

## 3. Check of the schedule code and of the checkpoint-buffer trap
Lists the lines marked `# MODIFIED` in `finetune.py` and shows why the schedule has to be set again after loading the checkpoint.

In [ ]:
print("Lines changed in the authors' schedule code (tagged '# MODIFIED'):\n")
!grep -n "MODIFIED" {SCRIPT}

# ---- tiny demo of the buffer trap (no GPU, no big model) ----
import torch.nn as nn
class Tiny(nn.Module):
    def __init__(self, abar):
        super().__init__(); self.register_buffer("abar", torch.tensor(abar)); self.w = nn.Linear(1, 1)

pretrained = Tiny([0.9, 0.5])           # "original schedule" stored in the checkpoint
saved = pretrained.state_dict()
model_B = Tiny([0.81, 0.25])            # we build the model with the MODIFIED schedule (abar ** 2)
print("\nbefore load_state_dict:", model_B.abar)
model_B.load_state_dict(saved)          # what DiffWater does when it loads the checkpoint
print("after  load_state_dict:", model_B.abar, "  <- the modified schedule is gone")

## 4. Settings shared by Run A and Run B

In [ ]:
EPOCHS = 10          # samples are written at epochs 2, 4, 6, 8, 10
BATCH  = 4
N_EVAL = 30          # test images in the final evaluation
N_SAMPLES = 3        # images sampled at each of the 5 epochs

## 5. Environment record

In [ ]:
import torch, sys, subprocess, hashlib
print("gpu:", torch.cuda.get_device_name(0)); print("torch:", torch.__version__); print("python:", sys.version.split()[0])
print("diffwater_commit:", subprocess.run(["git", "-C", "/content/DiffWater", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
print("finetune_md5:", hashlib.md5(open("/content/finetune.py", "rb").read()).hexdigest())

## 6. Launch (set RUN)
Checks the script md5, the DiffWater commit and the data fingerprints against the frozen values, then runs `finetune.py` in this cell. If Colab disconnects, run the setup, data and launch cells again in the same account: the job resumes from `resume.pt` and skips images that are already written.

In [ ]:
RUN = "B"                    # <-- "B" or "0" in the other accounts ("A" only to resume Run A). Nothing else differs.
assert RUN in ("0", "A", "B")

# ---- preflight: refuse to start unless this session IS the frozen experiment (experiment/EXPERIMENT.md) ----
EXPECT = dict(md5="38c97203e8ccafdfb80cb2be74b0bb3a", commit="7db68f34060bbf1d7c5b8153664fb16a3dcda1ae",
              train_fp="6a9f33de995c", split_fp="f8cc4714a30c")
tr = sorted(os.listdir(f"{OUT_TRAIN}/input_256"))
got = dict(
    md5=hashlib.md5(open(SCRIPT, "rb").read()).hexdigest(),
    commit=subprocess.run(["git", "-C", REPO_ROOT, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
    train_fp=hashlib.md5("".join(tr).encode()).hexdigest()[:12],
    split_fp=hashlib.md5(json.dumps(json.load(open(f"{A_DIR}/test_split_dark30.json"))).encode()).hexdigest()[:12])
check = [k for k in EXPECT if not (k == "train_fp" and RUN == "0")]     # Run 0 trains nothing
for k in check:
    print(f"{k:9s} {got[k]}   {'OK' if got[k] == EXPECT[k] else 'MISMATCH (expected ' + EXPECT[k] + ')'}")
bad = [k for k in check if got[k] != EXPECT[k]]
assert not bad, f"not the frozen experiment: {bad}. (wrong commit: git -C {REPO_ROOT} checkout {EXPECT['commit']})"
assert (EPOCHS, BATCH, N_EVAL, N_SAMPLES) == (10, 4, 30, 3), "settings differ from the frozen values (10, 4, 30, 3)"

OUT = f"{A_DIR}/runs/Run{RUN}"; os.makedirs(OUT, exist_ok=True)
already = subprocess.run('pgrep -f "[f]inetune.py"', shell=True, capture_output=True, text=True).stdout.strip()
assert not already, f"finetune.py is already running (pids {already.split()}) - not starting a second copy"

# ---- launch: blocking, so the cell stays busy and Colab sees an active session ----
train_arg = "" if RUN == "0" else f"--train_root {OUT_TRAIN} "          # Run 0 does no training
cmd = (f"exec python -u {SCRIPT} --run {RUN} --repo_dir {REPO_DIR} --ckpt_prefix {CKPT_PREFIX} "
       f"{train_arg}--eval_root {OUT_TEST} --out_dir {OUT} "
       f"--epochs {EPOCHS} --batch_size {BATCH} --n_eval {N_EVAL} --n_samples {N_SAMPLES}")
print(cmd)
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in p.stdout:
        if "UserWarning" in line or "lab2rgb" in line:                   # harmless colour-conversion warnings
            continue
        print(line, end="")
except KeyboardInterrupt:
    p.terminate()
    raise
finally:
    rc = p.wait()
print("exit code:", rc)
assert rc == 0, f"finetune.py exited with code {rc}"

## 7. Package the results of a finished run (optional)

In [ ]:
import zipfile
OUT = f"{A_DIR}/runs/Run{RUN}"
assert "ALL DONE" in open(f"{OUT}/run.log").read(), "This run has not finished yet (no 'ALL DONE' in run.log)."
ZIP = f"{A_DIR}/Run{RUN}_results.zip"
SKIP = ("resume.pt", "final_gen.pth", ".tmp")
n = 0
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            if f.endswith(SKIP): continue
            p = os.path.join(root, f); z.write(p, os.path.relpath(p, A_DIR)); n += 1
print(f"{n} files -> {ZIP}  ({os.path.getsize(ZIP) / 1e6:.0f} MB)")